# Oxford Tutorial - LLM 评估与部署 (v6.0)## Cell 1: Persona Prompt (导师人设)**You are an Oxford tutorial fellow in LLM 评估与部署 (LLM Evaluation & Deployment).**Tutorial rules (read aloud before each session):- Never give direct answers. Use Socratic questioning.- Act as HBS devil's advocate: challenge every claim with a counter-case.- Reject vague claims. If a student says "vLLM 更快", ask "凭什么? 在什么 GPU/QPS 下? 14-24x 是与什么 baseline 比?"- End each turn with a probing question.- Reference real libraries/datasets from this unit: deepeval BaseMetric/LLMTestCase/evaluate, langsmith @traceable, tiktoken, vLLM PagedAttention, 投机解码 draft model, MoE DeepSeek V3 671B/37B, RAGAS, AgentBench.- Rate limit: 1 session per day per unit (防依赖, 见 cell6).**Mastery goal**: After 4+ Socratic rounds, student can independently:1. Design a deepeval BaseMetric for marketing copy (4 dimensions)2. Wire langsmith @traceable + tiktoken cost monitoring3. Defend a vLLM/投机解码/MoE selection decision tree

## Cell 2: Pre-Tutorial Task (强制 Retrieval Practice)> 牛津 tutorial 前提: 学生先提交一段 essay/解题/方案。无 pre-tutorial work = tutorial 取消。> Retrieval practice (Karpicke 2008): 提取练习比重读有效 2x。**Pre-tutorial essay (提交后才进入 cell3 Socratic loop)**:用 300 字回答以下问题 (任选一):**Q-A**: 某团队用 MMLU 78 vs 75 选了模型 A 弃用 B 部署到营销文案生成, 上线后业务转化率下降。用 LLM 评估三层框架诊断问题, 给出正确做法。**Q-B**: 用 deepeval 自定义 `BaseMetric` 评估一段营销文案 "AI 神器一键搞定所有写作" 的无害性 (Harmlessness), 写出 `LLMTestCase` 字段 + `measure()` 返回的 `MetricMeasurement`。**Q-C**: DeepSeek V3 (MoE 671B/37B) 定价 gpt-4o 的 1/10, 某团队仍选 gpt-4o。列出 2 个非价格因素 + 说明 vLLM 自建 + 投机解码在什么场景能进一步降本。**提交方式**: 把 essay 字符串赋值给 `student_essay` 变量 (cell3 第 1 行)。LLM 仿真会读取并按 essay 内容分支追问。

In [ ]:
# Cell 3: Multi-turn Socratic Loop (静态 if/else 仿真, 不调真实 API)# 5+ 苏格拉底问: 为什么/反例/若前提变/凭什么/如何# 4+ 轮: 每轮根据 student_essay + student_response 分支追问import json, time# === 学生提交 pre-tutorial essay (填此处) ===student_essay = """某团队用 MMLU 选模型是错误的, 因为 MMLU 只测通用知识多选题, 不测营销文案能力。正确做法是自建评测集, 用 deepeval 跑四维度评估, 再用 langsmith 监控部署。"""# === student_model.json 初始化 (cell4 读写) ===student_model = {    "unit": "U-E3-D3",    "mastery": {"L1": 0, "L2": 0, "L3": 0, "L4": 0, "L5": 0},    "blind_spots": [],    "socratic_rounds_completed": 0,    "essay_topic": None,}def detect_essay_topic(essay):    if "MMLU" in essay or "三层框架" in essay:        return "Q-A"    if "BaseMetric" in essay or "LLMTestCase" in essay:        return "Q-B"    if "DeepSeek" in essay or "MoE" in essay or "投机解码" in essay:        return "Q-C"    return "unknown"student_model["essay_topic"] = detect_essay_topic(student_essay)# === 苏格拉底问库 (5+ 个, 静态) ===socratic_questions = {    "why":        "为什么你认为 MMLU 多选题不能反映营销文案能力? 凭什么? (追问因果)",    "counter":    "反例: 如果某模型 MMLU 90 分, 另一个 60 分, 你会完全不参考 MMLU 吗? (追问极端)",    "premise":    "若前提变: 如果团队没有标注人力建评测集, 你给的'正确做法'还成立吗? (追问鲁棒性)",    "basis":      "凭什么 deepeval 的 BaseMetric 比手写 print 评分更可信? 它的 measure() 返回什么? (追问机制)",    "how":        "如何用 langsmith @traceable 监控部署? 它装饰什么? 记录什么字段? (追问实操)",}# === 4 轮 Socratic loop (静态分支, 模拟追问) ===def socratic_loop(essay, topic):    rounds = []    if topic == "Q-A":        # Round 1: 为什么        rounds.append(("R1-WHY", socratic_questions["why"],            "学生应答: MMLU 测的是通用知识广度, 营销文案要测准确性/相关性/无害性/忠实性, 维度不同。"))        # Round 2: 反例        rounds.append(("R2-COUNTER", socratic_questions["counter"],            "学生应答: 会参考但不作为决定性指标, MMLU 用于初筛排除明显弱的模型, 但最终选型必须看任务评测集。"))        # Round 3: 若前提变        rounds.append(("R3-PREMISE", socratic_questions["premise"],            "学生应答: 无标注人力时可用 LLM-as-a-Judge (deepeval GEval) 自动评分, 但仍需 20-50 条人工种子样本校准。"))        # Round 4: 凭什么        rounds.append(("R4-BASIS", socratic_questions["basis"],            "学生应答: BaseMetric.measure() 返回 MetricMeasurement(score 0-1, reason 字符串), reason 可追溯评分依据。"))    elif topic == "Q-B":        rounds.append(("R1-WHY", "为什么 Harmlessness 要单独做维度, 不能并入准确性?",            "学生应答: 无害性管歧视/违规, 准确性管事实, 失败后果不同 (品牌危机 vs 误导)。"))        rounds.append(("R2-COUNTER", "反例: 一段文案事实准确但含性别歧视, 你的 BaseMetric 会给几分?",            "学生应答: 无害性直接 0 分, 准确性可能 5 分, 总分按权重 0.3*5+0.25*5+0.25*0+0.2*x。"))        rounds.append(("R3-PREMISE", "若营销文案面向不同地区, 无害性标准变吗?",            "学生应答: 变, 性别/种族/地域敏感词需按地区配置, BaseMetric 应支持 region 参数。"))        rounds.append(("R4-HOW", "如何让 measure() 的 reason 字段可追溯? 写什么进去?",            "学生应答: reason 写命中了哪个违规关键词 + 原文片段 + 修复建议。"))    elif topic == "Q-C":        rounds.append(("R1-WHY", "为什么 1/10 成本还不选 DeepSeek V3? 列非价格因素。",            "学生应答: 质量 (复杂推理弱于 gpt-4o) + 生态 (SDK/工具链成熟度) + 合规 (数据出境)。"))        rounds.append(("R2-COUNTER", "反例: 如果是简单分类任务, DeepSeek V3 仍输 gpt-4o 吗?",            "学生应答: 不输, 简单任务 DeepSeek V3 质量足够, 此时应选 DeepSeek 省 90% 成本。"))        rounds.append(("R3-PREMISE", "若数据不出境, 自建 vLLM + 投机解码, 什么场景能进一步降本?",            "学生应答: 高 QPS + 有配对 draft model + 延迟敏感场景, vLLM PagedAttention 14-24x 吞吐 + 投机解码 2-3x 加速。"))        rounds.append(("R4-BASIS", "凭什么 MoE 能 1/10 成本? 671B 总参但 37B 激活意味着什么?",            "学生应答: 单次推理只激活 37B 专家, 计算量 ≈ 37B 稠密模型, 但参数容量享 671B, 性价比高。"))    else:        rounds.append(("R1-WHY", socratic_questions["why"],            "学生应答: essay 主题未识别, 请重写 pre-tutorial essay。"))    return rounds# 执行 looprounds = socratic_loop(student_essay, student_model["essay_topic"])for rid, question, ideal_answer in rounds:    print(f"\n{'='*60}")    print(f"[{rid}] Oxford Tutor asks:")    print(f"  {question}")    print(f"[Ideal student answer]:")    print(f"  {ideal_answer}")    student_model["socratic_rounds_completed"] += 1print(f"\n{'='*60}")print(f"Socratic rounds completed: {student_model['socratic_rounds_completed']}")print(f"Essay topic detected: {student_model['essay_topic']}")

In [ ]:
# Cell 4: student_model.json 读写 (记录掌握度/盲点)# Hattie (2007) Visible Learning: student self-report + tutor observation 双源import json, osfrom pathlib import Path# === 评估掌握度 (基于 Socratic loop 表现) ===# 启发式: 若 essay 含关键词 + 4 轮全完成, 对应 ILO mastery +1essay_lower = student_essay.lower()mastery_map = {    "L1": any(k in student_essay for k in ["MMLU", "三层框架", "通用", "任务", "系统"]),    "L2": any(k in student_essay for k in ["BaseMetric", "deepeval", "measure", "LLMTestCase"]),    "L3": any(k in student_essay for k in ["langsmith", "@traceable", "tiktoken", "成本"]),    "L4": any(k in student_essay for k in ["vLLM", "投机解码", "MoE", "量化", "PagedAttention"]),    "L5": any(k in student_essay for k in ["DeepSeek", "671B", "37B", "1/10"]),}for ilo, hit in mastery_map.items():    if hit and student_model["socratic_rounds_completed"] >= 4:        student_model["mastery"][ilo] = 1  # 1 = 初步掌握, 需 AT 验证才到 80%# === 盲点检测 ===blind_spots = []if not mastery_map["L2"]:    blind_spots.append("deepeval BaseMetric measure() 机制未提及, 建议重做 practice.md Drill-01")if not mastery_map["L3"]:    blind_spots.append("langsmith @traceable + tiktoken 成本监控未提及, 建议重做 Drill-02")if not mastery_map["L4"]:    blind_spots.append("vLLM/投机解码/MoE 选型决策树未提及, 建议重做 Drill-03")if "reason" not in student_essay and "L2" in [k for k, v in mastery_map.items() if v]:    blind_spots.append("MetricMeasurement.reason 字段重要性未体现, 易踩坑")if not any(k in student_essay for k in ["AgentBench", "RAGAS"]):    blind_spots.append("AgentBench/RAGAS 等专用基准未提及, 选型初筛盲点")student_model["blind_spots"] = blind_spots# === 写入 student_model.json ===out_path = Path("student_model.json")with open(out_path, "w", encoding="utf-8") as f:    json.dump(student_model, f, ensure_ascii=False, indent=2)print(f"student_model.json written to: {out_path.absolute()}")print(f"\nMastery snapshot:")for ilo, val in student_model["mastery"].items():    status = "初步掌握" if val == 1 else "未掌握"    print(f"  {ilo}: {status}")print(f"\nBlind spots ({len(blind_spots)}):")for i, bs in enumerate(blind_spots, 1):    print(f"  {i}. {bs}")# === 读取验证 (round-trip) ===with open(out_path, "r", encoding="utf-8") as f:    loaded = json.load(f)assert loaded["unit"] == "U-E3-D3"assert loaded["socratic_rounds_completed"] >= 1print(f"\nRound-trip OK: {loaded['unit']}, {loaded['socratic_rounds_completed']} rounds")

## Cell 5: Hattie 4 级 Formative Feedback> Hattie & Timperley (2007) 4 级反馈: Task / Process / Self-Reg / Self。> Self 级 (表扬) 对学习效果 d=0.14 (低), 故本 tutorial 避免 Self 级表扬, 侧重前 3 级 + Feed-Forward。基于 cell4 的 student_model.json, 生成个性化反馈:

In [ ]:
# Cell 5: 生成 Hattie 4 级反馈 (基于 student_model.json)import jsonwith open("student_model.json", "r", encoding="utf-8") as f:    sm = json.load(f)topic = sm.get("essay_topic", "unknown")mastery = sm.get("mastery", {})blind = sm.get("blind_spots", [])# === [TASK] Task 级: 关于具体任务/答案的反馈 ===task_feedback = f"[TASK] 你的 pre-tutorial essay 主题={topic}。"if topic == "Q-A":    task_feedback += "你识别了 MMLU 不能直接选营销模型, 但未量化'通用 vs 任务'评测集的样本量差异 (100-500 条 vs MMLU 14000+ 题)。"elif topic == "Q-B":    task_feedback += "你提到了 BaseMetric, 但未写明 LLMTestCase 的 3 个必填字段 (input/actual_output/expected_output)。"elif topic == "Q-C":    task_feedback += "你列了非价格因素, 但未区分'质量差距可接受'的阈值 (notes.md § 关键回顾 4: <10% 即切便宜模型)。"else:    task_feedback += "essay 主题未识别, 请重写后重跑 cell3。"# === [PROCESS] Process 级: 关于解题过程/策略的反馈 ===process_feedback = "[PROCESS] 你的推理链长度="rounds_done = sm.get("socratic_rounds_completed", 0)process_feedback += f"{rounds_done}/4 轮。"if rounds_done >= 4:    process_feedback += "完整走完 4 轮 Socratic, 推理链健康。但每轮是否都给出 ideal answer? 若只是被动回答, 缺主动反问。"else:    process_feedback += "未完成 4 轮, 推理链断裂。建议回 practice.md Diagnostic 重做先测。"# === [SELF-REG] Self-Reg 级: 关于自我监控/调节的反馈 ===selfreg_feedback = "[SELF-REG] 你的盲点自检:"if not blind:    selfreg_feedback += "无检测到盲点, 但这可能是 essay 过于泛泛导致关键词命中假阳性。建议用 Feynman 技术向非技术同学复述 LLM 评估三层框架, 若对方听不懂即真实盲点。"else:    selfreg_feedback += f"检测到 {len(blind)} 个盲点。请用 24h 间隔重复 (非立即) 重做对应 drill, 间隔效应促固化。"# === [FEED-FORWARD] Feed-Forward 级: 下一步去哪 ===ff = "[FEED-FORWARD] 下一步:"weak_los = [ilo for ilo, v in mastery.items() if v == 0]if weak_los:    ff += f"优先补 {', '.join(weak_los)}。"    if "L2" in weak_los:        ff += " L2 对应 practice.md Drill-01 + starter.ipynb TODO2/3。"    if "L3" in weak_los:        ff += " L3 对应 Drill-02 + TODO4/5。"    if "L4" in weak_los:        ff += " L4 对应 Drill-03 + TODO6。"else:    ff += " 5 个 ILO 全初步掌握, 进入 practice.md P2 Milestone (deepeval 可跑 .ipynb)。"ff += " 24h 后再做 1 次 tutorial (限频, 见 cell6)。"# === 输出 (避免 Self 级表扬) ===print("=" * 60)print(task_feedback)print("=" * 60)print(process_feedback)print("=" * 60)print(selfreg_feedback)print("=" * 60)print(ff)print("=" * 60)

## Cell 6: 限频 + Exit Artifact### 限频 (防依赖)- **每单元 1 次/天**: tutorial.ipynb 每天最多跑 1 次。Oxford tutorial 的价值在于 pre-tutorial 准备 + 间隔反思, 不在于高频对话。- **24h 间隔**: 两次 tutorial 之间至少 24 小时。间隔效应 (Cepeda 2006) 促固化。- **禁止**: 用 tutorial 替代 practice.md 的 drill。drill 是刻意练习 (主动提取), tutorial 是苏格拉底追问 (被动澄清), 不可互换。- **依赖信号**: 若学生 1 周内跑 tutorial >3 次仍未达 mastery, 触发 practice.md § 7 Weak Loop (回退上一 drill + Worked Example), 而非继续刷 tutorial。### Exit Artifact (本次 tutorial 结束必交)> 若 blind_spots 为空且 5 ILO 全初步掌握, 可不交 exit artifact, 直接进入 P2 Milestone。> 否则必交以下 exit artifact:1. **2-3 个盲点** (从 cell4 student_model.json 的 blind_spots 选, 或自述新发现的盲点):   - 盲点 1: ___________________________________   - 盲点 2: ___________________________________   - 盲点 3: ___________________________________2. **推荐复习单元** (基于盲点, 从以下选 1-2 个):   - [ ] 重做 practice.md Drill-01 (deepeval BaseMetric 四维度)   - [ ] 重做 practice.md Drill-02 (langsmith @traceable + tiktoken 成本)   - [ ] 重做 practice.md Drill-03 (vLLM/投机解码/MoE 决策树)   - [ ] 重读 notes.md § 关键回顾 3 (四维度营销映射)   - [ ] 重读 notes.md § 2026 前沿 (LLM-as-a-Judge / deepeval / vLLM / 投机解码 / MoE)   - [ ] 重读 notes.md § 关键回顾 4 (模型选择决策框架)   - [ ] 做 reading.md 的 LLM-as-a-Judge / AgentBench 深链阅读3. **24h 后复测**: 间隔 24 小时后, 重跑 cell3 Socratic loop (用新 essay), 对比 mastery 是否提升。### Tutorial 结束检查清单- [ ] cell2 pre-tutorial essay 已写- [ ] cell3 Socratic loop 4 轮全完成- [ ] cell4 student_model.json 已写入并 round-trip 验证- [ ] cell5 Hattie 4 级反馈已读- [ ] cell6 exit artifact 已填 (或盲点为空已豁免)- [ ] 24h 间隔已记录 (下次 tutorial 最早时间: ____)---*v6.0 牛津 Tutorial LLM 仿真层。基于 Oxford tutorial system + HBS case method + Hattie (2007) 4 级反馈 + Karpicke (2008) retrieval practice。**Socratic loop 为静态 if/else 仿真, 不调真实 LLM API。*